# 02 — Create leakage-aware frozen splits
Builds transitive dependency components from software, repository, and publication identities. Produces approximately 70/15/15 train/validation/test partitions with seed 42 and verifies zero group leakage.


In [ ]:
from pathlib import Path
import subprocess, sys, json
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass
REPO=Path('/content/research_software_classification_attributes')
if not REPO.exists():
    subprocess.run(['git','clone','-b','data-finalization','https://github.com/kuefmz/research_software_classification_attributes.git',str(REPO)],check=True)
ROOT=Path('/content/drive/MyDrive/phd_research_software')
FROZEN=ROOT/'data/frozen'; SPLITS=ROOT/'splits'; SPLITS.mkdir(parents=True,exist_ok=True)


In [ ]:
for name in ['level1_high_level','level2_fine_grained']:
    data=FROZEN/f'{name}.jsonl'
    out=SPLITS/f'{name}_group_aware_seed42.csv'
    subprocess.run([sys.executable,str(REPO/'scripts/create_group_aware_splits.py'),'--data',str(data),'--out',str(out),'--seed','42'],check=True)
    audit=json.loads(out.with_suffix('.audit.json').read_text())
    assert audit['passed'] and audit['leakage_violation_count']==0
    print(name, json.dumps(audit,indent=2))
